# Triagem Clínico Geral × Especialista

Classificador binário para apoiar o encaminhamento de laudos entre atendimento
generalista e especializado, a partir do texto do laudo/observação médica.

**Objetivo geral:** validar a estratégia de modelagem (baseline vs. arquitetura
em duas etapas, threshold de decisão) em cima do dataset real do projeto
(`data/raw/laudos.csv`) antes de fixar o pipeline de produção (`modelo.joblib`).

Relatório visual consolidado com os mesmos resultados: `binary_triage/reports/report.html`.


## 1. Configuração do ambiente =========================

Nesta etapa iremos configurar o ambiente de trabalho, importar as bibliotecas
necessárias e fixar a seed de aleatoriedade. Reaproveitamos as funções de
`binary_triage/train.py` e `binary_triage/features.py` em vez de duplicar a
lógica de limpeza/modelagem aqui — isso garante que os números deste notebook
batem exatamente com o `modelo.joblib` gerado pelo script de treino oficial.

In [1]:

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name == "binary_triage" else str(Path.cwd()))

import re
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from binary_triage.train import (
    RAW_PATH, RANDOM_SEED, POSITIVE, NEGATIVE,
    load_and_clean, split_data, build_candidates, make_tfidf_only,
    evaluate, evaluate_from_proba, two_stage_proba, interpret_model,
)
from binary_triage.features import TextMetaFeatures, SPECIALTY_KEYWORD_SETS
from binary_triage.model_wrapper import ThresholdedBinaryClassifier

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_colwidth", 100)
plt.rcParams["figure.facecolor"] = "white"
BAR_COLOR = "#3b6ea5"


## 2. Carregamento dos dados =========================

### Dataset: `data/raw/laudos.csv`

Duas colunas apenas — não existem variáveis estruturadas independentes
(idade, sexo, duração, medicamentos como campos separados) neste dataset:

| Coluna | Tipo | Descrição |
|---|---|---|
| `texto` | texto livre | Resumo/observação clínica (laudo). Base do corpus é o *Medical Abstracts TC Corpus*, em inglês. |
| `especialidade` | categórica (5 valores) | Especialidade de destino: `clinica_geral`, `oncologia`, `cardiologia`, `neurologia`, `gastroenterologia`. É a coluna target — a única categórica de baixa cardinalidade, e o nome já indica o destino do encaminhamento. |

O **target binário** deste projeto é derivado de `especialidade`:
`clinica_geral -> CLINICO_GERAL`, as outras quatro -> `ESPECIALISTA`.

In [2]:

df_raw = pd.read_csv(RAW_PATH)
print(f"Shape: {df_raw.shape}")
df_raw.head(3)


Shape: (14438, 2)


,texto,especialidade
0,Tissue changes around loose prostheses. A canine model to investigate the effects of an antiinfl...,clinica_geral
1,Neuropeptide Y and neuron-specific enolase levels in benign and malignant pheochromocytomas. Neu...,oncologia
2,"Sexually transmitted diseases of the colon, rectum, and anus. The challenge of the nineties. Dur...",gastroenterologia


In [3]:

df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14438 entries, 0 to 14437
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   texto          14438 non-null  object
 1   especialidade  14438 non-null  object
dtypes: object(2)
memory usage: 225.7+ KB


Não há necessidade de conversão de tipos — as duas colunas já são `object`
(string). Criamos a coluna `target` a partir de `especialidade` e seguimos
para a análise exploratória.

In [4]:

df_raw["target"] = np.where(df_raw["especialidade"] == "clinica_geral", NEGATIVE, POSITIVE)
df_raw.head(3)


,texto,especialidade,target
0,Tissue changes around loose prostheses. A canine model to investigate the effects of an antiinfl...,clinica_geral,CLINICO_GERAL
1,Neuropeptide Y and neuron-specific enolase levels in benign and malignant pheochromocytomas. Neu...,oncologia,ESPECIALISTA
2,"Sexually transmitted diseases of the colon, rectum, and anus. The challenge of the nineties. Dur...",gastroenterologia,ESPECIALISTA


## 3. Análise Exploratória =========================

Nesta etapa iremos analisar o dataset em profundidade antes de treinar
qualquer modelo: valores ausentes, duplicidade/vazamento de rótulo, a
variável target, o tamanho das observações e os padrões linguísticos de cada
classe. Esta análise vai fundamentar as decisões de limpeza e de engenharia
de features das seções seguintes.

### 3.1 Valores ausentes

**Objetivo:** identificar valores ausentes e strings vazias antes de qualquer
processamento de texto.

In [5]:

print("Valores nulos por coluna:")
display(df_raw.isnull().sum())
print(f"Textos vazios/whitespace: {(df_raw['texto'].str.strip() == '').sum()}")


Valores nulos por coluna:


texto            0
especialidade    0
target           0
dtype: int64

Textos vazios/whitespace: 0


Zero nulos e zero textos vazios — o dataset já chega limpo nesse
aspecto. O problema de qualidade real está na duplicidade de rótulo, tratado
a seguir.

### 3.2 Duplicidade e vazamento de rótulo

**Objetivo:** verificar se existem linhas duplicadas e se essa duplicidade
representa risco de vazamento entre treino e teste — condição necessária
antes de decidir a estratégia de split (seção 5).

In [6]:

print(f"Linhas 100% duplicadas (texto + especialidade): {df_raw.duplicated().sum()}")

dupe_mask = df_raw.duplicated(subset=["texto"], keep=False)
dupes = df_raw[dupe_mask]
g_multi = dupes.groupby("texto")["especialidade"].nunique()
g_bin = dupes.groupby("texto")["target"].nunique()

print(f"Linhas envolvidas em texto duplicado (ignorando o rótulo): {dupe_mask.sum()}")
print(f"Grupos de texto duplicado com especialidade DIFERENTE entre as duplicatas: {(g_multi > 1).sum()}")
print(f"Grupos com TARGET BINÁRIO conflitante entre as duplicatas: {(g_bin > 1).sum()}")


Linhas 100% duplicadas (texto + especialidade): 0
Linhas envolvidas em texto duplicado (ignorando o rótulo): 6140
Grupos de texto duplicado com especialidade DIFERENTE entre as duplicatas: 2929
Grupos com TARGET BINÁRIO conflitante entre as duplicatas: 2411


2.929 textos idênticos aparecem com especialidade diferente entre
duplicatas — o corpus original mapeia categorias de doença a partir de
`condition_name`, e um mesmo abstract pode tocar mais de uma condição, mas o
dataset força rótulo único por linha. Em **2.411** desses grupos o conflito
chega a atingir o próprio target binário (mesmo texto ora `CLINICO_GERAL`, ora
`ESPECIALISTA`) — isso é ruído de rótulo irresolvível, não uma questão de
desempate, e por isso essas linhas serão **removidas** antes do split (não
depois), junto com duplicidade pura de texto. A função `load_and_clean()`
(`binary_triage/train.py`) implementa exatamente essa regra — reaproveitamos
aqui para manter os números idênticos ao pipeline de treino oficial.

In [7]:

df = load_and_clean()
print(f"Dataset limpo: {len(df)} linhas ({len(df_raw) - len(df)} removidas, {(len(df_raw)-len(df))/len(df_raw):.1%} do total)")


2026-08-16 20:37:15,528 INFO Limpeza: 14438 linhas originais -> 9344 após remover 5094 linhas com rótulo binário conflitante entre duplicatas do mesmo texto -> 8816 após deduplicar texto exato (528 linhas removidas por duplicidade pura).


Dataset limpo: 8816 linhas (5622 removidas, 38.9% do total)


### 3.3 Variável target

**Objetivo:** entender como a variável que o modelo vai prever se
comporta — balanceamento de classes e composição da classe ESPECIALISTA.

In [8]:

print("Distribuição multiclasse original (pós-limpeza):")
vc_multi = df["especialidade"].value_counts()
display(vc_multi)
display((vc_multi / len(df) * 100).round(1).astype(str) + "%")

print("\nDistribuição do target binário:")
vc_bin = df["target"].value_counts()
display(vc_bin)
display((vc_bin / len(df) * 100).round(1).astype(str) + "%")
print(f"\nRazão de desbalanceamento: {vc_bin.max() / vc_bin.min():.2f}x")


Distribuição multiclasse original (pós-limpeza):


especialidade
clinica_geral        2394
oncologia            2364
cardiologia          2081
neurologia           1183
gastroenterologia     794
Name: count, dtype: int64

especialidade
clinica_geral        27.2%
oncologia            26.8%
cardiologia          23.6%
neurologia           13.4%
gastroenterologia     9.0%
Name: count, dtype: object


Distribuição do target binário:


target
ESPECIALISTA     6422
CLINICO_GERAL    2394
Name: count, dtype: int64

target
ESPECIALISTA     72.8%
CLINICO_GERAL    27.2%
Name: count, dtype: object


Razão de desbalanceamento: 2.68x


In [9]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
vc_multi.plot(kind="barh", ax=axes[0], color=BAR_COLOR)
axes[0].set_title("Distribuição multiclasse (especialidade)")
axes[0].invert_yaxis()

vc_bin.plot(kind="bar", ax=axes[1], color=[BAR_COLOR, "#a5c4e0"])
axes[1].set_title("Distribuição do target binário")
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\3739064179.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


72,8% dos laudos são ESPECIALISTA contra 27,2% CLINICO_GERAL
(~2,7:1) — desbalanceamento moderado, tratável com `class_weight` e ajuste de
threshold (seções 6 e 7), sem necessidade de reamostragem.

### 3.4 Tamanho das observações

**Objetivo:** verificar se o tamanho do texto por si só já separa as classes
(o que seria um atalho barato para o modelo) e mapear a distribuição via
quantis.

In [10]:

df["n_chars"] = df["texto"].str.len()
df["n_words"] = df["texto"].str.split().str.len()

quantiles = [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
display(df.groupby("target")["n_chars"].quantile(quantiles).unstack().round(0))
display(df.groupby("target")[["n_chars", "n_words"]].mean().round(1))


,0.10,0.25,0.50,0.75,0.90,0.95,0.99
target,,,,,,,
CLINICO_GERAL,553.0,820.0,1150.0,1522.0,1806.0,2023.0,2559.0
ESPECIALISTA,574.0,879.0,1232.0,1607.0,1878.0,2052.0,2617.0


,n_chars,n_words
target,,
CLINICO_GERAL,1184.2,173.7
ESPECIALISTA,1250.5,182.9


In [11]:

fig, ax = plt.subplots(figsize=(7, 4))
for cls, color in [(NEGATIVE, "#a5c4e0"), (POSITIVE, BAR_COLOR)]:
    ax.hist(df.loc[df["target"] == cls, "n_chars"], bins=40, alpha=0.6, label=cls, color=color)
ax.set_xlabel("Caracteres por observação")
ax.set_ylabel("Frequência")
ax.legend()
ax.set_title("Distribuição de tamanho do texto por classe")
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\3410611034.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


As distribuições praticamente se sobrepõem (mediana ~1.180 caracteres em
ambas as classes) — tamanho de texto **não discrimina** a classe e não é um
atalho útil para o modelo. Ainda assim, entra como uma das features testadas
na seção 4, junto com contagem de termos por especialidade.

### 3.5 Padrões linguísticos por classe

**Objetivo:** identificar se cada especialidade tem vocabulário próprio, e se
a classe CLINICO_GERAL tem um padrão textual identificável ou se é de fato
uma classe "resíduo" (premissa central do problema, a validar empiricamente).

In [12]:

stop = set(
    "the a an of and to in with was were is are for on by as at be this that "
    "which from or not have has had it its patients patient study we results "
    "than these there also may can been between such among into two after "
    "using used all no both during but our their most however each other one "
    "cases case group groups significantly significant compared shown showed found"
    .split()
)

def top_terms(texts, n=12):
    words = []
    for t in texts:
        toks = re.findall(r"[a-zA-Z]{3,}", t.lower())
        words.extend(w for w in toks if w not in stop)
    return Counter(words).most_common(n)

for esp in ["clinica_geral", "oncologia", "cardiologia", "neurologia", "gastroenterologia"]:
    terms = top_terms(df.loc[df["especialidade"] == esp, "texto"])
    print(f"{esp:20s}: " + ", ".join(f"{w}({c})" for w, c in terms))


clinica_geral       : less(1087), treatment(914), disease(764), who(704), blood(696), more(662), associated(641), cells(611), clinical(608), acute(608), three(607), children(600)
oncologia           : cancer(2166), cell(2072), tumor(1975), cells(1848), carcinoma(1446), tumors(1321), treatment(1149), disease(1135), human(893), survival(889), therapy(795), breast(770)
cardiologia         : less(1928), pressure(1850), coronary(1755), blood(1737), ventricular(1375), disease(1351), heart(1281), artery(1200), left(1198), hypertension(1175), myocardial(1171), treatment(962)
neurologia          : disease(528), brain(477), treatment(442), clinical(415), cerebral(391), pain(375), more(355), less(330), normal(317), subjects(308), who(299), age(290)


gastroenterologia   : disease(752), less(503), liver(394), treatment(384), gastric(275), normal(264), bowel(242), more(239), who(230), associated(225), three(211), hepatitis(210)


`clinica_geral` não tem vocabulário próprio claro — os termos mais
frequentes (`disease`, `treatment`, `clinical`, `years`) são genéricos e
compartilhados com as outras quatro classes. As especialidades, em contraste,
têm vocabulário claramente distintivo (`cancer`/`carcinoma`/`tumor` para
oncologia, `coronary`/`cardiac`/`hypertension` para cardiologia, etc.). Isso
**confirma empiricamente** a premissa do problema: Clínico Geral é a classe
residual, sem padrão textual único — e por isso dividi-la em subcategorias
específicas (respiratória, dermatológica, preventiva, ver seção 8) **não é
sustentado pelos dados**: são abstracts acadêmicos sobre doenças, não relatos
de queixa de paciente com esse recorte.

## 4. Engenharia de features =========================

Sem colunas estruturadas independentes, derivamos features numéricas do
próprio texto (`binary_triage/features.py::TextMetaFeatures`): tamanho
(chars/words), densidade de dígitos, nº de sentenças, e contagem de termos
associados a cada especialidade — listas construídas a partir da seção 3.5,
não inventadas. Testamos empiricamente na seção 6 se combinar essas features
com TF-IDF melhora o modelo.

In [13]:

meta_preview = TextMetaFeatures().transform(df["texto"].head(5))
pd.DataFrame(meta_preview, columns=TextMetaFeatures().get_feature_names_out())


,n_chars,n_words,avg_word_len,n_digits,n_sentences,kw_oncologia,kw_cardiologia,kw_neurologia,kw_gastroenterologia,kw_total_especialista,kw_any_especialista
0,1056.0,156.0,5.692308,2.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1206.0,177.0,5.401130,21.0,12.0,6.0,0.0,2.0,0.0,8.0,1.0
2,1770.0,254.0,5.795276,6.0,13.0,2.0,0.0,0.0,5.0,7.0,1.0
3,560.0,78.0,6.038462,2.0,5.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1522.0,220.0,5.622727,24.0,11.0,0.0,1.0,0.0,0.0,1.0,1.0


## 5. Preparação para modelagem =========================

**Objetivo:** dividir treino/validação/teste sem vazamento. Split
estratificado por `especialidade` (granularidade mais fina que o target
binário), 70/15/15. Como o dataset já foi deduplicado por texto na seção 3.2,
cada linha é um documento único — a preocupação de vazamento por duplicidade
já foi resolvida na origem, antes do split (não precisamos de um split por
grupo/paciente adicional).

In [14]:

train_df, val_df, test_df = split_data(df)

X_train, y_train = train_df[["texto"]], train_df["target"]
X_val, y_val = val_df[["texto"]], val_df["target"]
X_test, y_test = test_df[["texto"]], test_df["target"]

for name, part in [("treino", train_df), ("validação", val_df), ("teste", test_df)]:
    dist = (part["target"].value_counts(normalize=True) * 100).round(1).to_dict()
    print(f"{name:12s} n={len(part):5d}  {dist}")


2026-08-16 20:37:16,206 INFO Split -> treino: 6171 (70.0%) | validação: 1322 (15.0%) | teste: 1323 (15.0%)


2026-08-16 20:37:16,208 INFO   Distribuição target (treino): {'ESPECIALISTA': 72.8, 'CLINICO_GERAL': 27.2}


2026-08-16 20:37:16,208 INFO   Distribuição target (validação): {'ESPECIALISTA': 72.8, 'CLINICO_GERAL': 27.2}


2026-08-16 20:37:16,209 INFO   Distribuição target (teste): {'ESPECIALISTA': 72.9, 'CLINICO_GERAL': 27.1}


treino       n= 6171  {'ESPECIALISTA': 72.8, 'CLINICO_GERAL': 27.2}
validação    n= 1322  {'ESPECIALISTA': 72.8, 'CLINICO_GERAL': 27.2}
teste        n= 1323  {'ESPECIALISTA': 72.9, 'CLINICO_GERAL': 27.1}


## 6. Modelagem e comparação de candidatos =========================

### 6.1 Candidatos

- `tfidf_lr` / `tfidf_lr_balanced` — TF-IDF (1-2 grams) + LogisticRegression, com e sem `class_weight="balanced"`
- `tfidf_linearsvc_calibrated` — TF-IDF + LinearSVC, calibrado (sigmoid) para gerar `predict_proba`
- `lsa_embeddings_lr` — TF-IDF + TruncatedSVD (200 componentes, "embedding" denso via LSA) + LogisticRegression. Optamos por LSA em vez de embeddings de transformer porque este ambiente não tem acesso a download de modelo pré-treinado nem GPU — é a alternativa de representação densa mais direta disponível localmente.
- `tfidf_meta_combined_lr` — TF-IDF + features estruturadas da seção 4, via `ColumnTransformer`
- `two_stage_multiclass` — classificador multiclasse (5 especialidades); a probabilidade de ESPECIALISTA é derivada como `1 - p(clinica_geral)`

Todos usam `Pipeline`/`ColumnTransformer` treinados **só no conjunto de
treino** — o vetorizador nunca vê texto de validação/teste antes da hora.

In [15]:

candidates = build_candidates()
results = {}
fitted = {}
for name, pipe in candidates.items():
    pipe.fit(X_train, y_train)
    metrics, _, _ = evaluate(pipe, X_val, y_val)
    results[name] = metrics
    fitted[name] = pipe

stage1 = make_tfidf_only(LogisticRegression(max_iter=2000, random_state=RANDOM_SEED))
stage1.fit(X_train, train_df["especialidade"])
p_pos_val_2stage = two_stage_proba(stage1, X_val)
metrics_2stage, _ = evaluate_from_proba(p_pos_val_2stage, y_val)
results["two_stage_multiclass"] = metrics_2stage

comparison_df = pd.DataFrame(results).T.sort_values("f1_macro", ascending=False)
comparison_df.round(4)


,precision_especialista,recall_especialista,f1_especialista,precision_clinico,recall_clinico,f1_clinico,f1_macro,roc_auc,pr_auc
tfidf_lr_balanced,0.9108,0.8380,0.8729,0.6422,0.7799,0.7044,0.7887,0.8963,0.9578
tfidf_linearsvc_calibrated,0.8621,0.9221,0.8911,0.7432,0.6045,0.6667,0.7789,0.9011,0.9599
tfidf_meta_combined_lr,0.9072,0.8120,0.8570,0.6065,0.7772,0.6813,0.7692,0.8801,0.9521
two_stage_multiclass,0.8265,0.9699,0.8925,0.8490,0.4540,0.5917,0.7421,0.8978,0.9568
tfidf_lr,0.7873,0.9875,0.8761,0.8947,0.2841,0.4313,0.6537,0.8955,0.9572
lsa_embeddings_lr,0.7604,0.9886,0.8596,0.8429,0.1643,0.2751,0.5673,0.8490,0.9376


In [16]:

fig, ax = plt.subplots(figsize=(8, 4))
order = comparison_df.index
colors = [BAR_COLOR if m == "tfidf_lr_balanced" else "#c9c9c9" for m in order]
ax.barh(order, comparison_df.loc[order, "f1_macro"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("F1-macro (validação)")
ax.set_title("Comparação de candidatos")
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\804677312.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Vencedor por F1-macro: `tfidf_lr_balanced`. `LinearSVC` calibrado fica
muito próximo. `lsa_embeddings_lr` tem o pior desempenho — reduzir a
dimensionalidade via SVD perde justamente os termos raros e específicos
(nomes de doença) que mais discriminam a classe Especialista neste corpus.

### 6.2 Arquitetura em duas etapas vs. classificador direto

**Objetivo:** decidir se vale a pena manter um classificador multiclasse
intermediário (seção 8 do enunciado do projeto) ou se um classificador
binário direto já é suficiente. Comparação justa: ambos calibrados no mesmo
ponto de operação (mesmo recall de Especialista), não no threshold padrão 0,5.

In [17]:

direct = fitted["tfidf_lr_balanced"]
p_direct_val = direct.predict_proba(X_val)[:, list(direct.classes_).index(POSITIVE)]

def closest_at_recall(p, target_recall):
    best_t, best_diff, best_m = None, 999, None
    for t in np.arange(0.05, 0.96, 0.01):
        m, _ = evaluate_from_proba(p, y_val, threshold=round(t, 2))
        diff = abs(m["recall_especialista"] - target_recall)
        if diff < best_diff:
            best_diff, best_t, best_m = diff, round(t, 2), m
    return best_t, best_m

t_direct, m_direct_matched = closest_at_recall(p_direct_val, 0.9377)
t_2stage, m_2stage_matched = closest_at_recall(p_pos_val_2stage, 0.9377)
print(f"direct    @ threshold={t_direct}: precision={m_direct_matched['precision_especialista']:.4f}  recall={m_direct_matched['recall_especialista']:.4f}  f1_macro={m_direct_matched['f1_macro']:.4f}")
print(f"two_stage @ threshold={t_2stage}: precision={m_2stage_matched['precision_especialista']:.4f}  recall={m_2stage_matched['recall_especialista']:.4f}  f1_macro={m_2stage_matched['f1_macro']:.4f}")


direct    @ threshold=0.4: precision=0.8575  recall=0.9377  f1_macro=0.7807
two_stage @ threshold=0.55: precision=0.8563  recall=0.9408  f1_macro=0.7806


No threshold padrão (0,5), a two-stage parece muito melhor em recall
(~0,97 vs ~0,84) — mas isso é só efeito de não estar calibrada. No mesmo
ponto de recall (~0,94), a precisão **empata** entre as duas abordagens. A
arquitetura extra (treinar e manter um classificador multiclasse) não compra
ganho de desempenho aqui — fica o modelo direto, mais simples de treinar e
servir em produção.

## 7. Calibração de threshold =========================

**Objetivo:** encontrar o ponto de corte de probabilidade certo para este
problema — 0,50 não é ele. Falso negativo (Especialista classificado como
Clínico Geral) é o erro caro aqui: risco de atraso diagnóstico. Falso positivo
custa só tempo de especialista. Critério: entre os thresholds com recall
Especialista ≥ 90%, escolher o que maximiza F1-macro.

In [18]:

proba_val = direct.predict_proba(X_val)
p_pos_val = proba_val[:, list(direct.classes_).index(POSITIVE)]

threshold_rows = []
for t in np.arange(0.30, 0.71, 0.05):
    m, _ = evaluate_from_proba(p_pos_val, y_val, threshold=round(t, 2))
    m["threshold"] = round(t, 2)
    threshold_rows.append(m)
threshold_df = pd.DataFrame(threshold_rows).set_index("threshold")
display(threshold_df.round(4))

safe = threshold_df[threshold_df["recall_especialista"] >= 0.90]
final_threshold = safe["f1_macro"].idxmax() if len(safe) else threshold_df["recall_especialista"].idxmax()
print(f"Threshold final escolhido: {final_threshold}")


,precision_especialista,recall_especialista,f1_especialista,precision_clinico,recall_clinico,f1_clinico,f1_macro,roc_auc,pr_auc
threshold,,,,,,,,,
0.30,0.7853,0.9875,0.8749,0.8919,0.2758,0.4213,0.6481,0.8963,0.9578
0.35,0.8210,0.9668,0.8879,0.8298,0.4345,0.5704,0.7292,0.8963,0.9578
0.40,0.8575,0.9377,0.8958,0.7770,0.5822,0.6656,0.7807,0.8963,0.9578
0.45,0.8854,0.8982,0.8918,0.7159,0.6880,0.7017,0.7967,0.8963,0.9578
0.50,0.9108,0.8380,0.8729,0.6422,0.7799,0.7044,0.7887,0.8963,0.9578
0.55,0.9358,0.7715,0.8458,0.5833,0.8579,0.6945,0.7701,0.8963,0.9578
0.60,0.9477,0.6771,0.7898,0.5095,0.8997,0.6506,0.7202,0.8963,0.9578
0.65,0.9656,0.5826,0.7267,0.4575,0.9443,0.6164,0.6715,0.8963,0.9578
0.70,0.9870,0.4746,0.6410,0.4109,0.9833,0.5796,0.6103,0.8963,0.9578


Threshold final escolhido: 0.4


In [19]:

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(threshold_df.index, threshold_df["recall_especialista"], marker="o", color="#c9622a", label="Recall Especialista")
ax.plot(threshold_df.index, threshold_df["precision_especialista"], marker="o", color=BAR_COLOR, label="Precision Especialista")
ax.axvline(final_threshold, color="#999999", linestyle="--", label=f"threshold escolhido = {final_threshold}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.legend()
ax.set_title("Precision × Recall (Especialista) por threshold — validação")
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\2620345997.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Avaliação final =========================

**Objetivo:** medir o desempenho real do modelo em dados nunca vistos. Refit
do modelo escolhido em treino+validação combinados (mais dado para o modelo
de produção), avaliação única e não enviesada no teste.

In [20]:

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

final_pipeline = build_candidates()["tfidf_lr_balanced"]
final_pipeline.fit(X_trainval, y_trainval)

test_metrics, y_pred_test, p_pos_test = evaluate(final_pipeline, X_test, y_test, threshold=final_threshold)
print(classification_report(y_test, y_pred_test, digits=4))
pd.Series(test_metrics).round(4)


               precision    recall  f1-score   support

CLINICO_GERAL     0.7455    0.5794    0.6520       359
 ESPECIALISTA     0.8554    0.9263    0.8894       964

     accuracy                         0.8322      1323
    macro avg     0.8004    0.7529    0.7707      1323
 weighted avg     0.8256    0.8322    0.8250      1323



precision_especialista    0.8554
recall_especialista       0.9263
f1_especialista           0.8894
precision_clinico         0.7455
recall_clinico            0.5794
f1_clinico                0.6520
f1_macro                  0.7707
roc_auc                   0.8899
pr_auc                    0.9550
dtype: float64

In [21]:

cm = confusion_matrix(y_test, y_pred_test, labels=[NEGATIVE, POSITIVE])
fig, ax = plt.subplots(figsize=(4.5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=[NEGATIVE, POSITIVE]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Matriz de confusão — teste (threshold={final_threshold:.2f})")
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\2491654457.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Recall de Especialista de 92,6% no teste, muito próximo dos 93,8%
observados em validação — sem sinal de overfit no threshold escolhido.

## 9. Sub-classificação por especialidade =========================

**Objetivo:** a decisão binária (seções 1-8) só diz "precisa de especialista"
— não diz qual. Isso é pouco acionável numa triagem real, então treinamos um
segundo classificador, **só nas linhas ESPECIALISTA**, para responder "qual
especialidade" entre oncologia, cardiologia, neurologia e gastroenterologia.
As duas decisões são independentes: mesmo split de treino/val/teste, mas o
sub-modelo nunca vê linhas CLINICO_GERAL — ele não compete com a decisão
binária, só a complementa.

In [22]:

esp_train = train_df[train_df["target"] == POSITIVE]
esp_val = val_df[val_df["target"] == POSITIVE]
esp_test = test_df[test_df["target"] == POSITIVE]
esp_trainval = pd.concat([esp_train, esp_val])

print("Distribuição de especialidade (treino, apenas ESPECIALISTA):")
display(esp_train["especialidade"].value_counts())

specialty_pipeline = make_tfidf_only(
    LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)
)
specialty_pipeline.fit(esp_trainval[["texto"]], esp_trainval["especialidade"])

y_pred_specialty = specialty_pipeline.predict(esp_test[["texto"]])
print(classification_report(esp_test["especialidade"], y_pred_specialty, digits=4))


Distribuição de especialidade (treino, apenas ESPECIALISTA):


especialidade
oncologia            1655
cardiologia          1456
neurologia            828
gastroenterologia     556
Name: count, dtype: int64

                   precision    recall  f1-score   support

      cardiologia     0.9476    0.8658    0.9048       313
gastroenterologia     0.7674    0.8319    0.7984       119
       neurologia     0.7512    0.9040    0.8205       177
        oncologia     0.9375    0.8873    0.9117       355

         accuracy                         0.8766       964
        macro avg     0.8509    0.8723    0.8589       964
     weighted avg     0.8856    0.8766    0.8788       964



In [23]:

cm_specialty = confusion_matrix(esp_test["especialidade"], y_pred_specialty, labels=list(specialty_pipeline.classes_))
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ConfusionMatrixDisplay(cm_specialty, display_labels=specialty_pipeline.classes_).plot(
    ax=ax, cmap="Oranges", colorbar=False, xticks_rotation=45
)
ax.set_title("Matriz de confusão — sub-classificação por especialidade (teste)")
fig.tight_layout()
plt.show()


C:\Users\Patrick\AppData\Local\Temp\ipykernel_10304\668036193.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


87,7% de acurácia entre as 4 especialidades (macro F1 0,86) — bem acima
do baseline de 35% (classe majoritária, oncologia). A confusão mais comum é
cardiologia↔neurologia e oncologia↔neurologia, plausível: sintomas
cardiovasculares e neurológicos (dor torácica, tontura) e achados oncológicos
inespecíficos podem se sobrepor no vocabulário do abstract. O modelo final
(`modelo.joblib`) embute os dois classificadores: `predict()` decide
CLINICO_GERAL/ESPECIALISTA, e `predict_detailed()` adiciona a especialidade
mais provável quando a decisão é ESPECIALISTA.

## 10. Interpretabilidade e análise de erros =========================

**Objetivo:** entender por que o modelo classifica cada laudo como classifica,
e caracterizar os casos em que ele erra.

In [24]:

lines = interpret_model(final_pipeline, "tfidf_lr_balanced")
print("\n".join(lines[:45]))


# Interpretabilidade — tfidf_lr_balanced


## Top termos associados a ESPECIALISTA (coeficiente positivo)

- `tfidf__cancer`: +5.2476
- `tfidf__tumor`: +4.4572
- `tfidf__carcinoma`: +3.9507
- `tfidf__hypertension`: +3.6648
- `tfidf__tumors`: +3.6235
- `tfidf__myocardial`: +2.8265
- `tfidf__coronary`: +2.6630
- `tfidf__brain`: +2.3236
- `tfidf__hypertensive`: +2.1410
- `tfidf__heart`: +2.0470
- `tfidf__cholesterol`: +2.0424
- `tfidf__melanoma`: +2.0317
- `tfidf__disease`: +1.9866
- `tfidf__aortic`: +1.9643
- `tfidf__diarrhea`: +1.9324
- `tfidf__pressure`: +1.9017
- `tfidf__blood pressure`: +1.8989
- `tfidf__cerebral`: +1.8546
- `tfidf__breast`: +1.8413
- `tfidf__left`: +1.8124
- `tfidf__malignant`: +1.8096
- `tfidf__seizures`: +1.7563
- `tfidf__leukemia`: +1.7297
- `tfidf__lesions`: +1.7060
- `tfidf__heart failure`: +1.6613
- `tfidf__metastases`: +1.6183
- `tfidf__stroke`: +1.6008
- `tfidf__hepatitis`: +1.5803
- `tfidf__carcinomas`: +1.5575
- `tfidf__gastric`: +1.5471

## Top termos ass

In [25]:

test_view = test_df.copy().reset_index(drop=True)
test_view["y_pred"] = y_pred_test
test_view["p_especialista"] = p_pos_test

fn = test_view[(test_view["target"] == POSITIVE) & (test_view["y_pred"] == NEGATIVE)]
fp = test_view[(test_view["target"] == NEGATIVE) & (test_view["y_pred"] == POSITIVE)]
print(f"Falsos negativos: {len(fn)} / {len(test_view)}  |  Falsos positivos: {len(fp)} / {len(test_view)}")

print("\n--- Exemplo de falso negativo (real=ESPECIALISTA, previsto=CLINICO_GERAL) ---")
row = fn.sample(1, random_state=RANDOM_SEED).iloc[0]
print(f"especialidade original: {row['especialidade']} | p(ESPECIALISTA)={row['p_especialista']:.3f}")
print(row["texto"][:300])


Falsos negativos: 71 / 1323  |  Falsos positivos: 151 / 1323

--- Exemplo de falso negativo (real=ESPECIALISTA, previsto=CLINICO_GERAL) ---
especialidade original: neurologia | p(ESPECIALISTA)=0.393
Sensorineural hearing loss: a reversible effect of valproic acid. We report 2 patients over the age of 70 who, while on valproate (VPA) for complex partial seizures, developed sensorineural hearing loss. Following discontinuation of VPA for nonaudiologic reasons, the patients reported improved heari


O falso negativo acima ilustra o limite do modelo baseado em texto:
sintoma e causa não usam vocabulário tipicamente associado à especialidade de
destino — o caso "parece" genérico até a suspeita clínica entrar em jogo,
algo que só o texto do abstract não captura bem.

## 11. Conclusão =========================

- **Target**: derivado de `especialidade` (`clinica_geral` → `CLINICO_GERAL`,
  demais → `ESPECIALISTA`), único caminho suportado pela estrutura do dataset
  (só 2 colunas, sem variáveis estruturadas independentes).
- **Limpeza**: ~39% das linhas brutas removidas por conflito de rótulo entre
  duplicatas ou duplicidade pura de texto — sem essa etapa haveria vazamento
  garantido entre treino e teste.
- **Modelo escolhido (decisão binária)**: `tfidf_lr_balanced` (TF-IDF +
  LogisticRegression, `class_weight="balanced"`). A arquitetura em duas etapas
  (multiclasse → binário) não trouxe ganho real quando comparada no mesmo
  ponto de operação — descartada em favor da solução mais simples.
- **Sub-classificação por especialidade**: quando o modelo decide
  ESPECIALISTA, um segundo classificador (treinado só nas linhas ESPECIALISTA)
  aponta qual das 4 especialidades — 87,7% de acurácia, muito acima do
  baseline de classe majoritária (~35%). Sem isso, "precisa de especialista"
  não seria acionável numa triagem real.
- **Threshold final**: calibrado para priorizar recall de Especialista (erro
  de falso negativo é o mais custoso clinicamente), mantendo F1-macro
  competitivo.
- **Limitações**: corpus em inglês (abstracts acadêmicos), não generaliza
  para laudos reais em português sem re-treino; `clinica_geral` aqui é o
  balde residual "general pathological conditions" do corpus original, não
  uma amostra de atendimento de rotina. Ferramenta de apoio à triagem — não
  substitui avaliação médica.

Modelo final salvo em `binary_triage/modelo.joblib` por `train.py` (script de
treino oficial, embute os dois classificadores — binário e por especialidade);
inferência via `binary_triage/predict.py`; artefatos completos em
`binary_triage/reports/` (matriz de confusão binária e por especialidade,
curva PR, comparação de modelos, interpretabilidade, exemplos de erro) e
relatório visual consolidado em `binary_triage/reports/report.html`.